In [2]:
from huggingface_hub import HfApi
from huggingface_hub import hf_hub_download
from datasets import load_dataset
import pandas as pd

import os
import json

token = os.environ.get("HF_TOKEN")
assert token is not None, "HF_TOKEN is not set"


/home/jeromeku/kernels/kernel-dev/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [31]:

api = HfApi(token=token)
# lighteval repo
repo_id = "jeromeku/details_meta-llama__Llama-3.2-1B-Instruct_private"

# lm-evals repo
repo_name = "jeromeku/lm_evals_details-private"

In [51]:
# Get samples from lm-evals repo
samples_file = hf_hub_download(repo_id=repo_name, filename="meta-llama__Llama-3.2-1B-Instruct/samples_mmlu_high_school_geography_2025-04-04T09-59-14.088683.jsonl", token=token, repo_type="dataset")
data = []
with open(samples_file, "r") as f:
    for line in f:
        data.append(json.loads(line))
samples_df = pd.DataFrame(data)

# Get details from lighteval repo

In [52]:
details_filename = "2025-04-04T10-58-23.820704/details_original|mmlu:high_school_geography|0_2025-04-04T10-58-23.820704.parquet"
details_file = hf_hub_download(repo_id=repo_id, filename=details_filename, token=token, repo_type="dataset")
details_df = pd.read_parquet(details_file)

In [53]:
details_df.head()

,choices,cont_tokens,example,full_prompt,gold,gold_index,input_tokens,instruction,metrics,num_asked_few_shots,num_effective_few_shots,padded,pred_logits,prediction_logits,predictions,specifics,truncated
0,"[A, B, C, D]","[[32, 33, 34, 35]]",The following are multiple choice questions (w...,The following are multiple choice questions (w...,[],[2],"[[128000, 791, 2768, 527, 5361, 5873, 4860, 32...",The following are multiple choice questions (w...,{'acc': 1},0,0,[1],[],[],"[-15.8125, -14.6875, -11.6875, -14.625]",None,[0]
1,"[A, B, C, D]","[[32, 33, 34, 35]]",The following are multiple choice questions (w...,The following are multiple choice questions (w...,[],[1],"[[128000, 791, 2768, 527, 5361, 5873, 4860, 32...",The following are multiple choice questions (w...,{'acc': 0},0,0,[1],[],[],"[-13.875, -13.5, -14.4375, -13.125]",None,[0]
2,"[A, B, C, D]","[[32, 33, 34, 35]]",The following are multiple choice questions (w...,The following are multiple choice questions (w...,[],[2],"[[128000, 791, 2768, 527, 5361, 5873, 4860, 32...",The following are multiple choice questions (w...,{'acc': 0},0,0,[9],[],[],"[-15.4375, -13.1875, -13.75, -14.9375]",None,[0]
3,"[A, B, C, D]","[[32, 33, 34, 35]]",The following are multiple choice questions (w...,The following are multiple choice questions (w...,[],[1],"[[128000, 791, 2768, 527, 5361, 5873, 4860, 32...",The following are multiple choice questions (w...,{'acc': 1},0,0,[0],[],[],"[-16.25, -12.5625, -15.8125, -16.5]",None,[0]
4,"[A, B, C, D]","[[32, 33, 34, 35]]",The following are multiple choice questions (w...,The following are multiple choice questions (w...,[],[3],"[[128000, 791, 2768, 527, 5361, 5873, 4860, 32...",The following are multiple choice questions (w...,{'acc': 0},0,0,[0],[],[],"[-15.75, -14.25, -15.0625, -14.8125]",None,[0]


In [54]:
from datasets import Dataset

samples_ds = Dataset.from_pandas(samples_df)

In [55]:
samples_ds

Dataset({
    features: ['doc_id', 'doc', 'target', 'arguments', 'resps', 'filtered_resps', 'filter', 'metrics', 'doc_hash', 'prompt_hash', 'target_hash', 'acc'],
    num_rows: 5
})

In [56]:
samples_ds.push_to_hub(repo_id="jeromeku/lm_evals_samples-private", token=token, private=True)

Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.95it/s]


CommitInfo(commit_url='https://huggingface.co/datasets/jeromeku/lm_evals_samples-private/commit/59330892947264dbcb1ef6521f8aeac3349b5ac4', commit_message='Upload dataset', commit_description='', oid='59330892947264dbcb1ef6521f8aeac3349b5ac4', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/jeromeku/lm_evals_samples-private', endpoint='https://huggingface.co', repo_type='dataset', repo_id='jeromeku/lm_evals_samples-private'), pr_revision=None, pr_num=None)

In [57]:
samples = load_dataset("jeromeku/lm_evals_samples-private")

Generating train split: 100%|██████████| 5/5 [00:00<00:00, 423.76 examples/s]


In [60]:
samples['train'][0]

{'doc_id': 0,
 'doc': {'answer': 1,
  'choices': ['a currency.',
   'a well-connected transportation infrastructure.',
   'government activity.',
   'a banking service.'],
  'question': 'The main factor preventing subsistence economies from advancing economically is the lack of',
  'subject': 'high_school_geography'},
 'target': '1',
 'arguments': {'gen_args_0': {'arg_0': 'The following are multiple choice questions (with answers) about high school geography.\n\nThe main factor preventing subsistence economies from advancing economically is the lack of\nA. a currency.\nB. a well-connected transportation infrastructure.\nC. government activity.\nD. a banking service.\nAnswer:',
   'arg_1': ' A'},
  'gen_args_1': {'arg_0': 'The following are multiple choice questions (with answers) about high school geography.\n\nThe main factor preventing subsistence economies from advancing economically is the lack of\nA. a currency.\nB. a well-connected transportation infrastructure.\nC. government ac